# Demo pipeline walkthrough

This notebook demonstrates the governed pipeline on synthetic fixtures only. It keeps all processing local and uses `MOCK_LLM=true` for deterministic example behavior.

In [ ]:
import json
import os
from pathlib import Path

os.environ['MOCK_LLM'] = 'true'
print('MOCK_LLM=true')
repo = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
print(repo)

## Synthetic inputs

The notebook loads synthetic fixtures from `tests/fixtures/` and existing example artifacts from `examples/`.

In [ ]:
fixture_dir = repo / 'tests' / 'fixtures'
example_dir = repo / 'examples'
fixture_names = sorted(p.name for p in fixture_dir.iterdir())
example_names = sorted(p.name for p in example_dir.iterdir())
fixture_names, example_names

## Pipeline stages covered

The code below shows the artifact chain for ingest, Pass 1, anchor handling, lens lock, Pass 2, grounding check, review, and audit emit without touching participant data.

In [ ]:
synthetic_transcript = json.loads((fixture_dir / 'synthetic_transcript_P01.json').read_text())
synthetic_pass1 = json.loads((fixture_dir / 'synthetic_pass1_output.json').read_text())
synthetic_anchor = json.loads((fixture_dir / 'synthetic_pass1_anchor.json').read_text())
synthetic_pass2 = json.loads((fixture_dir / 'synthetic_pass2_output.json').read_text())
synthetic_lens = json.loads((fixture_dir / 'synthetic_lens.json').read_text())
summary = {
    'ingest_fixture': synthetic_transcript.get('dataset_id'),
    'pass1_run': synthetic_pass1.get('run_label'),
    'anchor_type': synthetic_anchor.get('anchor_type'),
    'lens_locked': synthetic_lens.get('locked'),
    'pass2_claims': len(synthetic_pass2.get('claims', [])),
}
summary

## Example stage commands

These are the local commands a researcher would run in order.

In [ ]:
stage_commands = [
    'python src/agent/orchestrator.py --check-preflight',
    'python src/agent/pass1_runner.py --deid-path artifacts/deidentified_P01_session1.json --dataset-id SYNTHETIC_P01',
    'python src/modules/osf_uploader.py --anchor artifacts/pass1_anchor_SYNTHETIC_P01.json --doi https://osf.io/example1',
    'python src/modules/lens_dialogue.py --lock --run-id LENS_RUN_001 --researcher-id https://orcid.org/0000-0000-0000-0000',
    'python src/agent/pass2_runner.py --deid artifacts/deidentified_P01_session1.json --lens artifacts/lens_LENS_RUN_001.json --pass1-hash HASH --dataset-id SYNTHETIC_P01',
    'python src/modules/grounding_checker.py --pass2-output artifacts/pass2_output_seed42_SYNTHETIC_P01.json',
    'REVIEWER_ID=https://orcid.org/0000-0000-0000-0000 python src/tools/review_cli.py --dir artifacts/',
    'python src/modules/audit_emitter.py --dataset-id SYNTHETIC_P01 --run-id LENS_RUN_001',
]
for command in stage_commands:
    print(command)

## Auditability notes

This demo is intentionally minimal. It keeps outputs cleared, uses synthetic content, and documents how the governed artifact chain is expected to flow before a real local run.